In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Stichprobebasierte Quantendiagonalisierung vun enem Chemie-Hamiltonian

*Nůtzungsschätzung: ungefähr en Menutt op enem Heron r2 Prozessor (HENWIES: Dat es nur en Schätzung. Ding Laufzick kann variiere.)*
## Hengerjrond
En däm Tutorial zeije mer, wi verrauschte Quantestichprove nohbearbeidt wääde, öm en Approximation vum Jrondzustand vum Stickstoffmolekül $\text{N}_2$ bei Jliichjevechsbindungslänge ze fenge. Dobi bruche mer dä [stichprobebasierte Quantendiagonalisierungsalgorithmus (SQD)](https://arxiv.org/abs/2405.05068) för Stichprove us enem 59-Qubit-Quanteschaltkreis (52 System-Qubits + 7 Ancilla-Qubits). En Qiskit-basierte Implementierung es verfüjbar en dä [SQD Qiskit Addons](https://github.com/Qiskit/qiskit-addon-sqd), mieh Details fingste en dä entsprechende [Dokumentation](/guides/qiskit-addons-sqd) met enem [einfache Beispill](/guides/qiskit-addons-sqd-get-started) zom Aanfange.

SQD es en Technik för't Fenge vun Eijewääte un Eijevektor vun Quanteoperatore, wi däm Hamiltonian vun enem Quantesystem, unger Verwendung vun Quante- un verteiltem klassischem Computing zesamme. Dat verteilte klassische Computing weed verdendt, öm Stichprove vun enem Quanteprozessor ze verarweite un en Ziel-Hamiltonian en enem durch se opjespannte Ungersruum ze projeziere un ze diagonalisiere. Dat ermöjlesch SQD, robust jejenövver durch Quanterausche verfälschte Stichprove ze sin un met jroße Hamiltonians ömzejohn, wi Chemie-Hamiltonians met Millione vun Wechselwerkungstermen, di jenseits vun dä Richwiedt exakter Diagonalisierungsmethode lieje. Ene SQD-basierte Workflow hätt foljende Schrette:

1. Wähl ene Schaltkreis-Ansatz un wend en op enem Quantecomputer op ene Referenzzustand aan (en däm Fall dä [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)-Zustand).
2. Sample Bitstrings us däm resultierende Quantezustand.
3. Föhr dat *selbstkonsistente Konfigurationswidderherstellungsverfahre* op dä Bitstrings us, öm di Approximation vum Jrondzustand ze krije.

SQD funktioneert bekanntemaße joot, wann dä Ziel-Eijezustand dünn besetzt es: Di Wellefunktion weed vun ener Menge vun Basiszuständ $\mathcal{S} = {|x\rangle }$ jedraare, dänne Jrüß nit exponenziell met dä Problemjrüß wächst.

### Quantechemie
Di Eijeschafte vun Moleküle wääde wietjehend durch di Struktur vun dä Elektrone en ihne bestömmt. Als fermionische Deilcher künne Elektrone met enem mathematische Formalismus, Zweitquantisierung jenannt, beschrevve wääde. Di Idee es, dat et en Aanzahl vun *Orbitale* jit, vun däne jedes entweder leer es oder vun enem Fermion besetzt weed. E System vun $N$ Orbitale weed durch ene Satz fermionischer Vernichtungsoperatore ${\hat{a}_p}_{p=1}^N$ beschrevve, di di fermionische Antikommutatorrelationen erfülle:

$$
\begin{align*}
\hat{a}_p \hat{a}_q + \hat{a}_q \hat{a}_p &= 0, \\
\hat{a}_p \hat{a}_q^\dagger + \hat{a}_q^\dagger \hat{a}_p &= \delta_{pq}.
\end{align*}
$$

Dä adjunjierte Operator $\hat{a}_p^\dagger$ weed Erzeujungsoperator jenannt.

Bes jetz hätt uns Darstellung dä Spin nit berücksichticht, dä en fundamentale Eijeschaf vun Fermione es. Beim Berücksichtige vum Spin kumme di Orbitale en Paare vör, *Raumorbitale* jenannt. Jedes Raumorbital besteiht us zwei *Spinorbitale*, vun däne eins met "Spin-$\alpha$" un eins met "Spin-$\beta$" bezeichnet weed. Mer schrieve dann $\hat{a}_{p\sigma}$ för dä Vernichtungsoperator, dä met däm Spinorbital met Spin $\sigma$ ($\sigma \in {\alpha, \beta}$) em Raumorbital $p$ assozieet es. Wann mer $N$ als Aanzahl vun dä Raumorbitale nemme, jitt et insjsammt $2N$ Spinorbitale. Dä Hilbert-Ruum vun däm System weed vun $2^{2N}$ orthonormale Basisvektor opjespannt, di met zweiteilige Bitstrings bezeichnet wääde: $\lvert z \rangle = \lvert z_\beta z_\alpha \rangle = \lvert z_{\beta, N} \cdots z_{\beta, 1} z_{\alpha, N} \cdots z_{\alpha, 1} \rangle$.

Dä Hamiltonian vun enem molekulare System kann jeschrevve wääde als

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

wobei di $h_{pr}$ un $h_{prqs}$ komplexe Zahle sin, di molekulare Integrale jenannt wääde un us dä Spezifikation vum Molekül met enem Computerprogramm berechnet wääde künne. En däm Tutorial berechne mer di Integrale met däm Softwarepaket [PySCF](https://pyscf.org/).

För Details doröver, wi dä molekulare Hamiltonian herjeleit weed, konsulteer e Lehrboch zor Quantechemie (zom Beispill *Modern Quantum Chemistry* vun Szabo un Ostlund). För en övverjeordnete Erklärung, wi Quantechemieprobleme op Quantecomputer afjebeld wääde, luur de dir di Vorlesung [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE&t=900) vun dä Qiskit Global Summer School 2024 aan.

### Local Unitary Cluster Jastrow (LUCJ) Ansatz
SQD bruch ene Quanteschaltkreis-Ansatz, us däm Stichprove jezore wääde künne. En däm Tutorial verwende mer dä [Local Unitary Cluster Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) Ansatz wejen singer Kombination us physikalischer Motivation un Hardware-Fründlichkeit.

Dä LUCJ-Ansatz es en spezialisierte Form vum aaljemeine Unitary Cluster Jastrow (UCJ) Ansatz, dä di Form hätt

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

wobei $\lvert \Phi_0 \rangle$ ene Referenzzustand es, off dä Hartree-Fock-Zustand, un di $\hat{K}_\mu$ un $\hat{J}_\mu$ di Form hann

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

wobei mer dä Deilcherzahloperator defineert hann

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

Dä Operator $e^{\hat{K}_\mu}$ es en Orbitalrotation, di met bekannte Algorithme en linearer Deef un unger Verwendung linearer Konnektivität implementeert wääde kann.

Di Implementierung vum $e^{i \mathcal{J}_k}$ Term vum UCJ-Ansatz erforderich entweder vollständije Konnektivität oder di Verwendung vun enem fermionische Swap-Netzwärk, wat för verrauschte präfehlertolerante Quanteprozessore met bejeschronkte Konnektivität en Herausforderung es. Di Idee vum *lokale* UCJ-Ansatz es, Dünnbesetztheidsbedinunge op di $\mathbf{J}^{\alpha\alpha}$- un $\mathbf{J}^{\alpha\beta}$-Matrize opzeerleje, di et ermöjleche, se en konstanter Deef op Qubit-Topologien met bejeschronkte Konnektivität ze implementiere. Di Bedinunge wääde durch en Leß vun Indizes spezifizieet, di aanzeije, welche Matrixenträj em övvere Dreieck vun null verscheede sin dörfe (do di Matrize symmetrisch sin, moss nur dat övvere Dreieck spezifizieet wääde). Dis Indizes künne als Orbitalpaare interpretieet wääde, di meteinander interajeere dörfe.

Betrach als Beispill e quadratisch Jitter-Qubit-Topologie. Mer künne di $\alpha$- un $\beta$-Orbitale en parallele Linien op däm Jitter platziere, wobei Verbindunge zwesche dä Linien "Sprosse" vun ener Leiterforrm belde, wi folch:

![Qubit-Zuordnungsdiagramm för dä LUCJ-Ansatz op enem quadratische Jitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/baad4e53-5bfd-4cb4-8027-2ec26d27ecdd.avif)

Bei däm Setup sin Orbitale met däm selve Spin met ener Linientopologie verbonge, während Orbitale met ungerscheidliche Spin verbonge sin, wann se dat selve Raumorbital deile. Dat erjibt di foljende Indexbedinunge för di $\mathbf{J}$-Matrize:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, \ldots, N-1}
\end{align*}
$$

Met angere Wööt: Wann di $\mathbf{J}$-Matrize nur an dä aanjejovvene Indizes em övvere Dreieck vun null verscheede sin, kann dä $e^{i \mathcal{J}_k}$ Term op ener quadratische Topologie ohne Verwendung vun Swap-Gates en konstanter Deef implementeert wääde. Natüürlich mäht dat Operleje vun soche Bedinunge op dä Ansatz en winniger usdrucksstark, sodass möjlicherwise mieh Ansatz-Widderholunge nüüdich sin.

Di IBM&reg; Hardware hätt en Heavy-Hex-Jitter-Qubit-Topologie, en welchem Fall mer e "Zickzack"-Muster verwende künne, dat onge darjestellt es. En däm Muster wääde Orbitale met däm selve Spin op Qubits met ener Linientopologie affjebeld (rude un blaue Kreise), un en Verbindung zwesche Orbitale ungerscheidliche Spins es an jedem 4. Raumorbital vorhande, wobei di Verbindung durch e Ancilla-Qubit (violette Kreise) ermöjlesch weed. En däm Fall sin di Indexbedinunge

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, 4, 8, \ldots (p \leq N-1)}
\end{align*}
$$

![Qubit-Zuordnungsdiagramm för dä LUCJ-Ansatz op enem Heavy-Hex-Jitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Selbstkonsistente Konfigurationswidderherstellung
Dat selbstkonsistente Konfigurationswidderherstellungsverfahre es dozo usjeleht, su vill Signal wi möjlich us verrauschte Quantestichprove russzoholle. Do dä molekulare Hamiltonian di Deilcherzahl un Spin-Z erhält, es et sinnvoll, ene Schaltkreis-Ansatz ze wähle, dä dis Symmetriene och erhält. Wann en op dä Hartree-Fock-Zustand aanjedendt weed, hätt dä resultierende Zustand em rauschfreie Fall en faste Deilcherzahl un Spin-Z. Doher sulle di Spin-$\alpha$- un Spin-$\beta$-Hälften vun jedem us däm Zustand jesampelte Bitstring dat selve [Hamming-Jewech](https://en.wikipedia.org/wiki/Hamming_weight) wi em Hartree-Fock-Zustand hann. Wejen däm Vorhandensinn vun Rausche en aktuelle Quanteprozessore wääde manch jemessene Bitstrings dis Eijeschaf verletze. En einfache Forrm vun dä Postselektion würd dis Bitstrings verwärfe, ävver dat es verschwenderisch, weil dis Bitstrings velleich noch e bessje Signal enthalde. Dat selbstkonsistente Widderherstellungsverfahre versök, ene Deil vun däm Signal en dä Nohbearbeitung widder herzestelle. Dat Verfahre es iterativ un bruch als Einjabe en Schätzung vun dä durchschnettliche Besetzunge vun jedem Orbital em Jrondzustand, di zuerst us dä Rohstichprove berechnet weed. Dat Verfahre weed en ener Schleif usjeföhrt, un jede Iteration hätt di foljende Schrette:

1. För jede Bitstring, dä di spezifiziete Symmetriene verletzt, flipp sing Bits met enem probabilistische Verfahre, dat dozo usjeleht es, dä Bitstring noher an di aktuelle Schätzung vun dä durchschnettliche Orbitalbesetzunge ze brenge, öm ene neue Bitstring ze krije.
2. Sammle all alde un neue Bitstrings, di di Symmetriene erfülle, un entnimm Deilmenge vun ener em Voruss jewählte faste Jrüß.
3. För jede Deilmenge vun Bitstrings projezier dä Hamiltonian en dä durch di entsprechende Basisvektor opjespannte Ungersruum (luur en dä [vörrije Avschnitt](#quantum-chemistry) för en Beschrievung vun dä Basisvektor) un berech en Jrondzustandsschätzung vum projeziete Hamiltonian op enem klassische Computer.
4. Aktualiseer di Schätzung vun dä durchschnettliche Orbitalbesetzunge met dä Jrondzustandsschätzung met dä niddrichste Energie.

### SQD-Workflow-Diagramm
Dä SQD-Workflow es em foljende Diagramm darjestellt:

![Workflow-Diagramm vum SQD-Algorithmus](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Aanforderunge
Stell sicher vör däm Beginne vun däm Tutorial, dat de Foljends installeert häss:

- Qiskit SDK v1.0 oder höher, met [Visualisierungs](https://docs.quantum.ibm.com/api/qiskit/visualization)-Ungerstötzung
- Qiskit Runtime v0.22 oder höher (`pip install qiskit-ibm-runtime`)
- SQD Qiskit Addon v0.11 oder höher (`pip install qiskit-addon-sqd`)
- ffsim v0.0.58 oder höher (`pip install ffsim`)
## Setup

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Schrett 1: Klassische Einjabe op e Quanteproblem affbelde
En däm Tutorial finge mer en Approximation vum Jrondzustand vum Molekül em Jliichjevech em cc-pVDZ-Basissatz. Zuerst spezifiziere mer dat Molekül un sing Eijeschafte.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Vör däm Konstruiere vum LUCJ-Ansatz-Schaltkreis föhre mer zunächs en CCSD-Berechnung en dä foljende Code-Zell durch. Di [$t_1$- un $t_2$-Amplituden](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) us dä Berechnung wääde verdendt, öm di Parameter vum Ansatz ze initialisiere.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Jetz verwende mer [ffsim](https://github.com/qiskit-community/ffsim), öm dä Ansatz-Schaltkreis ze erschaffe. Do uns Molekül ene jeschlossenschaliche Hartree-Fock-Zustand hätt, verwende mer di spin-balanzierte Variante vum UCJ-Ansatz, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Mer övverjävve Wechselwerkungspaare, di för en Heavy-Hex-Jitter-Qubit-Topologie jeeijet sin (luur en dä [Hengerjrondavschnitt zom LUCJ-Ansatz](#local-unitary-cluster-jastrow-lucj-ansatz)). Mer setze `optimize=True` en dä `from_t_amplitudes`-Methode, öm di "komprimiete" Doppelfaktorisierung vun dä $t_2$-Amplituden ze ermöjleche (luur [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) en dä ffsim-Dokumentation för Details).

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


Mer empfehle di foljende Schrette, öm dä Ansatz ze optimiere un hardware-kompatibel ze maache.

- Wähl physikalische Qubits (`initial_layout`) us dä Ziel-Hardware us, di däm vörher beschrevvene "Zickzack"-Muster entspräche. Dat Aanleeje vun Qubits en däm Muster föhrt zo enem effizienten hardware-kompatible Schaltkreis met winniger Gates. Hee föje mer Code en, öm di Uswahl vum "Zickzack"-Muster ze automatisiere, dä en Bewertungsheuristik verdendt, öm di met däm usjewählte Layout verbongene Fehler ze minimiere.
- Jenerieer ene jestuffte Pass-Manager met dä Funktion [generate_preset_pass_manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) vun Qiskit met dinger Wahl vun `backend` un `initial_layout`.
- Setz di `pre_init`-Stuuf vun dingem jestuffte Pass-Manager op `ffsim.qiskit.PRE_INIT`. `ffsim.qiskit.PRE_INIT` enthält Qiskit-Transpiler-Päss, di Gates en Orbitalrotationen zerleeje un dann di Orbitalrotationen zesammeföhre, wat zo winniger Gates em endjültige Schaltkreis föhrt.
- Föhr dä Pass-Manager op dingem Schaltkreis us.
<details>
<summary>Code för automatisiete Uswahl vum "Zickzack"-Layout</summary>

</details>

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

## Schrett 3: Usföhre met Qiskit-Primitive
Noh dä Optimierung vum Schaltkreis för di Hardware-Usföhrung sin mer parat, en op dä Ziel-Hardware uszföhre un Stichprove för di Jrondzustandsenergieabschätzung ze sammle. Do mer nur ene Schaltkreis hann, verwende mer dä [Job-Usföhrungsmodus](/guides/execution-modes) vun Qiskit Runtime un föhre unse Schaltkreis us.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>